# 3.2.1 Price Dynamics and Volatility

This notebook is intentionally **load-only**. All heavy simulations and precomputations must be generated remotely on `gpu.enst.fr`, then copied back into `data/processed/remote_results/`.

Expected remote inputs:

- `price_dynamics_metadata.json`
- `plot_window_series.parquet`
- `volatility_summary.csv`
- `volatility_summary_aggregated.csv`

If any required file is missing, the notebook stops immediately with a clear error.

## 1. Setup

The figures and tables in this notebook are built only from compact remote artifacts. No calibration, no simulation, and no cache-building are performed locally.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "data/processed/remote_results"
META_FILE = RESULTS_DIR / "price_dynamics_metadata.json"
PLOT_FILE = RESULTS_DIR / "plot_window_series.parquet"
VOL_FILE = RESULTS_DIR / "volatility_summary.csv"
VOL_AGG_FILE = RESULTS_DIR / "volatility_summary_aggregated.csv"

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing precomputed file {path}. Run the remote precompute script on gpu.enst.fr.")
    return path

metadata = json.loads(require_file(META_FILE).read_text())
plot_df = pd.read_parquet(require_file(PLOT_FILE))
plot_df["timestamp"] = pd.to_datetime(plot_df["timestamp"])
vol_df = pd.read_csv(require_file(VOL_FILE))
vol_agg_df = pd.read_csv(require_file(VOL_AGG_FILE))

MODEL_ORDER = ["Real", "QRU", "QR", "FTQR", "SAQR"]
MODEL_COLORS = {
    "Real": "#111111",
    "QRU": "#4c78a8",
    "QR": "#f58518",
    "FTQR": "#54a24b",
    "SAQR": "#b279a2",
}
PERIOD_ORDER = ["Full day", "Calm", "Active"]

summary_df = pd.DataFrame(
    {
        "plot_day": [metadata["plot_day"]],
        "plot_window": [f"{metadata['plot_start']} to {metadata['plot_end']}"] ,
        "stats_days": [len(metadata["stats_days"])],
        "models": [", ".join(metadata["models"])],
        "annualization_factor": [metadata["annualization_factor"]],
    }
)
display(summary_df)


## 2. One-Day Visual Comparison

The representative-day plot window is precomputed remotely. The top panel reproduces the mid-price comparison; the bottom panel shows the 10-minute rolling annualized volatility from the same remote artifact.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for model in MODEL_ORDER:
    sub = plot_df[plot_df["model"] == model].copy()
    axes[0].plot(sub["timestamp"], sub["mid_price"], label=model, color=MODEL_COLORS[model], linewidth=1.8)
axes[0].set_title(f"Mid-Price Dynamics on {metadata['plot_day']} ({metadata['plot_start']}–{metadata['plot_end']})")
axes[0].set_ylabel("Mid-price")
axes[0].grid(alpha=0.2)
axes[0].legend(ncol=5, frameon=False)

for model in MODEL_ORDER:
    sub = plot_df[plot_df["model"] == model].copy()
    axes[1].plot(sub["timestamp"], sub["rolling_vol"], label=model, color=MODEL_COLORS[model], linewidth=1.8)
axes[1].set_title("Rolling Annualized Volatility (10-minute window, 1-second sampling)")
axes[1].set_ylabel("Annualized volatility")
axes[1].set_xlabel("Time")
axes[1].grid(alpha=0.2)
fig.tight_layout()

window_vol = (
    vol_df[(vol_df["day"] == metadata["plot_day"]) & (vol_df["period"] == "Calm")]
    [["model", "sigma_real", "sigma_sim", "relative_difference_pct", "quadratic_error"]]
    .sort_values("model")
)
display(window_vol.style.format({
    "sigma_real": "{:.4f}",
    "sigma_sim": "{:.4f}",
    "relative_difference_pct": "{:+.2f}",
    "quadratic_error": "{:.6f}",
} ))

best_match = (
    window_vol.assign(abs_gap=lambda x: (x["sigma_sim"] - x["sigma_real"]).abs())
    .sort_values("abs_gap")
    .iloc[0]["model"]
)
display(Markdown(
    f"""
### Interpretation

- The representative day and plot window were fixed remotely and stored in the metadata file.
- The plotted series are already synchronized on the same 1-second calendar grid.
- In this window, **{best_match}** is the closest match to the real market in volatility level.
"""
))


## 3. Multi-Day Volatility Benchmark

The benchmark table below is computed remotely from the same precomputed 1-second mid-price files. It reports average relative differences and average quadratic errors by period and model.

In [ ]:
vol_agg_df["period"] = pd.Categorical(vol_agg_df["period"], categories=PERIOD_ORDER, ordered=True)
vol_agg_df["model"] = pd.Categorical(vol_agg_df["model"], categories=["QRU", "QR", "FTQR", "SAQR"], ordered=True)
vol_agg_df = vol_agg_df.sort_values(["period", "model"]).reset_index(drop=True)

display(
    vol_agg_df.style.format(
        {
            "avg_relative_difference_pct": "{:+.2f}",
            "avg_quadratic_error": "{:.6f}",
            "mean_sigma_real": "{:.4f}",
            "mean_sigma_sim": "{:.4f}",
        }
    )
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, period in zip(axes, PERIOD_ORDER):
    sub = vol_agg_df[vol_agg_df["period"] == period].copy()
    ax.bar(sub["model"].astype(str), sub["avg_relative_difference_pct"], color=[MODEL_COLORS[m] for m in sub["model"].astype(str)])
    ax.axhline(0.0, color="black", linewidth=1)
    ax.set_title(period)
    ax.set_ylabel("Average relative difference (%)")
    ax.grid(axis="y", alpha=0.2)
fig.tight_layout()

best_by_period = vol_agg_df.loc[vol_agg_df.groupby("period")["avg_quadratic_error"].idxmin(), ["period", "model"]]
best_text = "\n".join([f"- **{row.period}**: {row.model}" for row in best_by_period.itertuples(index=False)])
display(Markdown(
    f"""
### Interpretation

- Best model by average quadratic error:
{best_text}
- These statistics are purely loaded from the remote result tables; the notebook does not rerun any simulation locally.
"""
))


## 4. Summary

This notebook is now a report-only view over remote artifacts. The laptop workload is limited to loading compact files, rendering figures, and formatting tables. All calibration, simulation, and benchmarking must be done remotely before opening the notebook.